In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from scipy.signal import savgol_filter

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------
# Load data
# -----------------------------
def load_data():

    print("Loading data...")

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


# -----------------------------
# Prepare spectral features
# -----------------------------
def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    print("Spectral features:", X.shape[1])

    return X, y, X_test


# -----------------------------
# Savitzky–Golay smoothing
# -----------------------------
def apply_savgol(X, X_test):

    print("Applying Savitzky–Golay smoothing...")

    X_sg = savgol_filter(
        X,
        window_length=11,
        polyorder=2,
        deriv=0,
        axis=1
    )

    X_test_sg = savgol_filter(
        X_test,
        window_length=11,
        polyorder=2,
        deriv=0,
        axis=1
    )

    return X_sg, X_test_sg


# -----------------------------
# Scale features
# -----------------------------
def scale_features(X, X_test):

    print("Scaling features...")

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    return X_scaled, X_test_scaled


# -----------------------------
# ElasticNet with Stratified CV
# -----------------------------
def elasticnet_stratified_cv(X, y, X_test):

    print("\nRunning ElasticNet Stratified CV...")

    # Bin target variable for stratification
    bins = pd.qcut(y, q=8, labels=False, duplicates="drop")

    skf = StratifiedKFold(n_splits=8, shuffle=True, random_state=42)

    alpha = 0.00207
    l1_ratio = 0.95

    print("Using parameters:")
    print("alpha =", alpha)
    print("l1_ratio =", l1_ratio)

    test_preds = np.zeros(len(X_test))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, bins)):

        print(f"\nFold {fold+1}")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=20000,
            tol=1e-3,
            selection="random",
            random_state=42
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, preds))

        print("Fold RMSE:", rmse)

        fold_scores.append(rmse)

        # accumulate test predictions
        test_preds += model.predict(X_test) / skf.n_splits

    print("\nFinal CV RMSE:", np.mean(fold_scores))

    return test_preds


# -----------------------------
# Save submission
# -----------------------------
def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    name = "exp19_elasticnet_stratified_savgol"

    path = f"../submissions/{name}.csv"

    submission.to_csv(path, index=False, header=False)

    print("\nSaved submission:", path)

    print(pd.read_csv(path, header=None).head())


# -----------------------------
# Pipeline
# -----------------------------
def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X, X_test = apply_savgol(X, X_test)

    X, X_test = scale_features(X, X_test)

    preds = elasticnet_stratified_cv(X, y, X_test)

    save_submission(test, preds)


main()

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Spectral features: 1555
Applying Savitzky–Golay smoothing...
Scaling features...

Running ElasticNet Stratified CV...
Using parameters:
alpha = 0.00207
l1_ratio = 0.95

Fold 1
Fold RMSE: 14.176723652961867

Fold 2
Fold RMSE: 12.057841134650634

Fold 3
Fold RMSE: 13.5803661836153

Fold 4
Fold RMSE: 11.967672066432883

Fold 5
Fold RMSE: 12.580342416172096

Fold 6
Fold RMSE: 10.922762299734025

Fold 7
Fold RMSE: 13.074214942686956

Fold 8
Fold RMSE: 13.095974762152391

Final CV RMSE: 12.681987182300768

Saved submission: ../submissions/exp19_elasticnet_stratified_savgol.csv
    0           1
0  95  145.847207
1  96  141.947372
2  97  155.134211
3  98  155.407266
4  99  152.650371
